# 23. __new__, initialization, and subclass hooks

**Level:** Advanced

**Learning outcome:** Validate immutable built-in construction and enforce extension invariants with __init_subclass__.

**Prerequisites:** 05-06 and 21: inheritance, MRO, and class-level behavior.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](22_Cached_Properties_and_Invalidation.ipynb) | [Next notebook](24_Async_Objects_Concurrency_and_Thread_Safety.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Advanced

**Learning goal:** Understand descriptors, slots, caching, __new__, and class-creation hooks without overusing them.

**Descriptors** define attribute behavior through methods such as __get__, __set__, and
__set_name__. Properties and ordinary bound methods rely on descriptor mechanisms. A
reusable validating descriptor can enforce the same invariant across several fields. Access
through the class should normally return the descriptor itself when instance is None.

**Attribute lookup** is richer than 'instance dictionary then class'. Data descriptors can
take precedence over instance dictionary entries; non-data descriptors such as ordinary
methods participate differently. __getattr__ is a fallback for missing attributes.
__getattribute__ intercepts ordinary attribute access and is easy to make recursively broken.
Prefer a property or explicit method unless interception is truly required.

**__slots__** declares a set of attribute storage names for instances and can remove their
ordinary __dict__ when the hierarchy permits it. It is not a security boundary, deep
immutability, or a universal performance fix. A base class with a dictionary can change the
result. Weak references and inheritance require deliberate design.

**Cached properties** compute once and store a result. They suit expensive derived values only
when invalidation semantics are clear. A cached value can become stale if inputs change.
Concurrent access may also compute a value more than once, so do not rely on exactly-once
side effects.

**Creation hooks.** __new__ creates instances and is useful when subclassing immutable built-ins.
__init_subclass__ customizes subclass creation without requiring a custom metaclass.
A metaclass controls class creation itself; `type(SomeClass)` is its metaclass. Most network
automation should prefer ordinary classes and explicit registries over metaclass machinery.

**Pause and explain:** Choose a property before a descriptor unless reuse justifies it; explain cache invalidation.

## Worked-example structure

This notebook contains **2 worked patterns and 20 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 079: Validate an immutable value in __new__

**Core concept:** An immutable built-in subclass must choose its value during instance creation.

**Network-engineering purpose:** A small lab VLAN value can behave numerically while rejecting invalid construction.

### Read the implementation in this order
1. Override __new__ on an int subclass.
2. Validate before creating the immutable integer.
3. Delegate construction to super().__new__.
4. Check numeric behavior and invalid-value rejection.

**Common mistake / boundary:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

**Practice after the ten worked scenarios:** Use composition instead and compare equality semantics with ordinary integers.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0781 - Validate an immutable value in __new__: branch scenario

**Scenario 01/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `1` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [1]:
# Example 0781: Validate an immutable value in __new__ - branch
case = {'vlan': 1}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
1


### Example 0782 - Validate an immutable value in __new__: campus scenario

**Scenario 02/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `10` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
10
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [2]:
# Example 0782: Validate an immutable value in __new__ - campus
case = {'vlan': 10}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
10


### Example 0783 - Validate an immutable value in __new__: dc scenario

**Scenario 03/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `100` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
100
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [3]:
# Example 0783: Validate an immutable value in __new__ - dc
case = {'vlan': 100}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
100


### Example 0784 - Validate an immutable value in __new__: wan scenario

**Scenario 04/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `200` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
200
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [4]:
# Example 0784: Validate an immutable value in __new__ - wan
case = {'vlan': 200}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
200


### Example 0785 - Validate an immutable value in __new__: lab scenario

**Scenario 05/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `999` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
999
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [5]:
# Example 0785: Validate an immutable value in __new__ - lab
case = {'vlan': 999}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
999


### Example 0786 - Validate an immutable value in __new__: wireless scenario

**Scenario 06/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `1000` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
1000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [6]:
# Example 0786: Validate an immutable value in __new__ - wireless
case = {'vlan': 1000}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
1000


### Example 0787 - Validate an immutable value in __new__: remote scenario

**Scenario 07/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `2000` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
2000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [7]:
# Example 0787: Validate an immutable value in __new__ - remote
case = {'vlan': 2000}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
2000


### Example 0788 - Validate an immutable value in __new__: core scenario

**Scenario 08/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `3000` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
3000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [8]:
# Example 0788: Validate an immutable value in __new__ - core
case = {'vlan': 3000}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
3000


### Example 0789 - Validate an immutable value in __new__: mgmt scenario

**Scenario 09/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `4000` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
4000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [9]:
# Example 0789: Validate an immutable value in __new__ - mgmt
case = {'vlan': 4000}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
4000


### Example 0790 - Validate an immutable value in __new__: backup scenario

**Scenario 10/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `4093` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
4093
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [10]:
# Example 0790: Validate an immutable value in __new__ - backup
case = {'vlan': 4093}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
4093


## Pattern 080: Register subclasses with __init_subclass__

**Core concept:** A subclass creation hook can enforce an extension invariant.

**Network-engineering purpose:** A controlled plugin family can require unique short names at definition time.

### Read the implementation in this order
1. Define the registry on a local base class.
2. Accept a named class-creation keyword.
3. Call super's hook and reject duplicate names.
4. Check the concrete class was registered and inspect its metaclass.

**Common mistake / boundary:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

**Practice after the ten worked scenarios:** Write the same registry using an explicit register call and compare discoverability.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0791 - Register subclasses with __init_subclass__: branch scenario

**Scenario 01/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [11]:
# Example 0791: Register subclasses with __init_subclass__ - branch
case = {'mtu': 1500}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0792 - Register subclasses with __init_subclass__: campus scenario

**Scenario 02/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [12]:
# Example 0792: Register subclasses with __init_subclass__ - campus
case = {'mtu': 9000}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0793 - Register subclasses with __init_subclass__: dc scenario

**Scenario 03/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [13]:
# Example 0793: Register subclasses with __init_subclass__ - dc
case = {'mtu': 9216}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0794 - Register subclasses with __init_subclass__: wan scenario

**Scenario 04/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [14]:
# Example 0794: Register subclasses with __init_subclass__ - wan
case = {'mtu': 576}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0795 - Register subclasses with __init_subclass__: lab scenario

**Scenario 05/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [15]:
# Example 0795: Register subclasses with __init_subclass__ - lab
case = {'mtu': 1600}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0796 - Register subclasses with __init_subclass__: wireless scenario

**Scenario 06/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [16]:
# Example 0796: Register subclasses with __init_subclass__ - wireless
case = {'mtu': 1500}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0797 - Register subclasses with __init_subclass__: remote scenario

**Scenario 07/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [17]:
# Example 0797: Register subclasses with __init_subclass__ - remote
case = {'mtu': 2000}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0798 - Register subclasses with __init_subclass__: core scenario

**Scenario 08/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [18]:
# Example 0798: Register subclasses with __init_subclass__ - core
case = {'mtu': 4470}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0799 - Register subclasses with __init_subclass__: mgmt scenario

**Scenario 09/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [19]:
# Example 0799: Register subclasses with __init_subclass__ - mgmt
case = {'mtu': 9100}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0800 - Register subclasses with __init_subclass__: backup scenario

**Scenario 10/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [20]:
# Example 0800: Register subclasses with __init_subclass__ - backup
case = {'mtu': 8192}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


## Check your understanding: __new__, initialization, and subclass hooks

**You should now be able to:** Validate immutable built-in construction and enforce extension invariants with __init_subclass__.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.